In [ ]:
import numpy as np
import torch 
from matplotlib import pyplot as plt
from utils import radonTransform
from MCMC.BPS_Gibbs_sparse import BPS_Gibbs
from matplotlib.pyplot import imread
from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio as PSNR
from skimage.metrics import structural_similarity as ssim

In [ ]:
height, width = 256,256
angleNum = 128

img = imread('95.png')
if img.shape[-1] <= 4:  # 如果是彩色图,即RGB或者RGBA,则通过取第一个通道转为灰度图。
    img = img[:, :, 0]
if img.dtype == 'uint8':
    img = img.astype('float32') / 255  # scale to [0, 1]
    
img = resize(img,(256,256))

In [ ]:
A = radonTransform(angleNum, height, width).astype('float32')

x = img.reshape(-1, 1).astype('float32')
y_noise_free = A @ x
sigma = 0.85
y = (y_noise_free + sigma * np.random.randn(*y_noise_free.shape)).astype('float32')

device1 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
hyper= [1,1,1,1]
x_init= torch.from_numpy(x).to(device1).view(height, width) + 1e-3 * torch.randn(height, width, device = device1)

In [ ]:
x_mean, x_std = BPS_Gibbs(x_init,torch.from_numpy(y.copy()).to(device1),torch.from_numpy(A.copy()).to(device1),sigma, hyper, gamma1 = 0, gamma2 = 1)

In [ ]:
pixel = 256
plt.subplot(2, 1, 1)
plt.imshow((x_mean.view(pixel, pixel)).cpu().numpy())
plt.colorbar(plt.imshow((x_mean.view(pixel, pixel)).cpu().numpy()))
plt.subplot(2, 1, 2)
plt.imshow((x_std.view(pixel, pixel)).cpu().numpy())
plt.colorbar(plt.imshow((x_std.view(pixel, pixel)).cpu().numpy()))
plt.show()